# 05 — Local Transformer sentiment fine-tuning

Prepared 9 November 2026 (Asia/Calcutta). **Generated, unexecuted notebook.**

Prepare a compact English Transformer experiment and inspect its eventual real checkpoint/training reports.

Inputs: ml/sentiment_transformer/config.yaml (genuine three-class) or configs/training/imdb_binary_transformer.yaml (separate binary benchmark), local licensed safetensors and tokenizer files.

Complete Phase 2 runtime acceptance and the relevant data/approval gates first. Both actual LR and SVM baseline reports must match the task and split hashes. No pretrained checkpoint is bundled.

No datasets, dependencies or model weights are fetched. No model is trained by running all cells with the default flag. Fill only real, approved local paths. Do not store private review text in committed outputs. Source presence is not Phase 2/3 acceptance.

Read [the Phase 3 runbook](../documentation/phase3/RUNBOOK.md) first.

In [ ]:
from pathlib import Path
import sys

# Explicit opt-in for later local inspection; this delivery has not executed cells.
RUN_APPROVED_EXPERIMENTS = False
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "ARCHITECTURE.md").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the D:/BDA_TA workspace.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workspace:", ROOT)
print("Experiment execution enabled:", RUN_APPROVED_EXPERIMENTS)

In [ ]:
CONFIG_PATH = ROOT / "ml/sentiment_transformer/config.yaml"
if RUN_APPROVED_EXPERIMENTS:
    from ml.common.config import load_config
    from ml.common.transformer_engine import preflight
    print(preflight(load_config(CONFIG_PATH), "sentiment_transformer"))
else:
    print("Preflight disabled; no model source or GPU runtime has been loaded.")

## Resource and run protocol

Start with sequence length 128, micro-batch 4 and accumulation 8; these are trials, not a guaranteed fit. Validate the current driver/PyTorch CUDA build using tiny tensor, forward/backward and AMP checks. BF16 requires a real executed probe; FP16 uses scaling. Do not silently move a planned CUDA training run onto CPU.

Prepare a separate approved maximum-updates-2 smoke config. Verify actual save/reload/resume before authorizing longer fine-tuning. One substantial model runs at a time. The shared loop records optimizer/scheduler/scaler/RNG and source/config/input-contract hashes, dynamic padding and bounded checkpoints.

## Manual execution later

```powershell
.\scripts\train_models.ps1 -Component sentiment_transformer -Action Preflight
.\scripts\train_models.ps1 -Component sentiment_transformer -Action Train -ConfigPath 'ml/sentiment_transformer/config.yaml' -ExecuteApprovedRun
```

Resume only an owned checkpoint with the identical reviewed config, manifests, base tokenizer and preprocessing contract:

```powershell
.\scripts\train_models.ps1 -Component sentiment_transformer -Action Resume -ConfigPath 'ml/sentiment_transformer/config.yaml' -ResumeCheckpoint '<owned checkpoint path>' -ExecuteApprovedRun
```

Use the actual checkpoint path saved by the run. No command in this notebook is executed automatically.

In [ ]:
RUN_DIRECTORY = None  # Real completed run below models/, after approved fitting.
if RUN_APPROVED_EXPERIMENTS and RUN_DIRECTORY:
    from ml.common.artifacts import read_json
    run = ROOT / RUN_DIRECTORY
    training_report = read_json(run / "training_report.json")
    model_manifest = read_json(run / "model_manifest.json")
    assert training_report["trained"] is True
    assert training_report["test_evaluated"] is False
    print({"task": training_report["task"], "updates": training_report["updates"],
           "tuning_history": training_report["tuning_history"],
           "calibration_metrics": training_report["calibration_metrics"],
           "peak_vram_allocated_bytes": training_report["peak_vram_allocated_bytes"],
           "experimental": model_manifest["experimental"]})
else:
    print("No trained run supplied; tuning, calibration and memory results are unmeasured.")

## Error analysis

Inspect actual truncation impact, class confusion, duplicate residuals, domain shift and neutral/mixed disagreements on permitted tuning examples. Do not log private text. The final model is selected on tuning macro-F1, calibrated on calibration and then evaluated once on the frozen test. Checkpoint source code alone is not evidence of successful resumption or deterministic equivalence.

## Interpretation and acceptance

Record measured results, source/model licenses, exact config and manifest hashes, code revision, dependency versions, hardware and unsupported cohorts after the approved runs. Keep the final held-out set separate from model selection. No result has been measured in this notebook. Update the model card from actual reports; missing evidence remains missing.